In [1]:
# Install required packages
!pip install -q transformers datasets torch accelerate

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import os
import shutil
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Configuration - IMPROVED
class Config:
    # Paths
    TRAIN_ZIP = '/kaggle/input/dimasr/subtask1'
    DATA_DIR = './subtask1_data'
    OUTPUT_DIR = './subtask1_outputs'
    CHECKPOINT_DIR = './subtask1_checkpoints'
    
    # Model - IMPROVED
    MODEL_NAME = 'bert-base-uncased'
    MAX_LENGTH = 128
    HIDDEN_DIM = 384  # Increased from 256
    DROPOUT = 0.2     # Reduced from 0.3 for better learning
    
    # Training - OPTIMIZED
    BATCH_SIZE = 64   # Increased from 32
    ACCUMULATION_STEPS = 1  # Set to 4 if OOM
    EPOCHS = 30       # Increased from 20
    LEARNING_RATE = 5e-5  # Increased from 2e-5
    WARMUP_RATIO = 0.15   # Increased from 0.1
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    
    # Early stopping - MORE PATIENT
    PATIENCE = 7  # Increased from 5
    
    # VA normalization
    VA_MIN = 1.0
    VA_MAX = 9.0
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    SEED = 42

config = Config()
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

# Copy files from input to working directory
files_to_copy = [
    "eng_laptop_train_alltasks.jsonl", "eng_laptop_dev_task1.jsonl",
    "eng_laptop_test_task1.jsonl", "eng_restaurant_train_alltasks.jsonl",
    "eng_restaurant_dev_task1.jsonl", "eng_restaurant_test_task1.jsonl"
]

for fname in files_to_copy:
    src = os.path.join(config.TRAIN_ZIP, fname)
    dst = os.path.join(config.DATA_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {fname}")

# Set seed
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)

print(f"\nDevice: {config.DEVICE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Hidden Dim: {config.HIDDEN_DIM}")
print(f"Dropout: {config.DROPOUT}")
print(f"Patience: {config.PATIENCE}")

Copied eng_laptop_train_alltasks.jsonl
Copied eng_laptop_dev_task1.jsonl
Copied eng_laptop_test_task1.jsonl
Copied eng_restaurant_train_alltasks.jsonl
Copied eng_restaurant_dev_task1.jsonl
Copied eng_restaurant_test_task1.jsonl

Device: cuda
Batch Size: 64
Learning Rate: 5e-05
Hidden Dim: 384
Dropout: 0.2
Patience: 7


In [4]:
# Data utility functions
def load_jsonl(file_path):
    """Load JSONL file"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def normalize_va(value):
    """Normalize VA from [1,9] to [0,1]"""
    return (value - config.VA_MIN) / (config.VA_MAX - config.VA_MIN)

def denormalize_va(value):
    """Denormalize VA from [0,1] to [1,9]"""
    return value * (config.VA_MAX - config.VA_MIN) + config.VA_MIN

def parse_va(va_string):
    """Parse VA string 'V.VV#A.AA' to (valence, arousal)"""
    v, a = va_string.split('#')
    return float(v), float(a)

def format_va(valence, arousal):
    """Format VA to 'V.VV#A.AA' with clipping and rounding"""
    v = np.clip(valence, config.VA_MIN, config.VA_MAX)
    a = np.clip(arousal, config.VA_MIN, config.VA_MAX)
    return f"{v:.2f}#{a:.2f}"

def extract_aspect_va_from_quadruplets(data):
    """
    IMPROVEMENT: Explicit conversion from Quadruplet to Aspect_VA format
    Filters out NULL aspects for cleaner training
    """
    processed = []
    for item in data:
        aspect_va_list = []
        for quad in item['Quadruplet']:
            if quad['Aspect'] != 'NULL':  # Skip implicit aspects
                aspect_va_list.append({
                    'Aspect': quad['Aspect'],
                    'VA': quad['VA']
                })
        
        if aspect_va_list:
            processed.append({
                'ID': item['ID'],
                'Text': item['Text'],
                'Aspect_VA': aspect_va_list
            })
    return processed

def highlight_aspect_in_text(text, aspect):
    """
    IMPROVEMENT: Add aspect position markers
    Helps model focus on relevant text regions
    """
    aspect_lower = aspect.lower()
    text_lower = text.lower()
    start_idx = text_lower.find(aspect_lower)
    
    if start_idx == -1:
        # Aspect not found in text, use explicit format
        return f"{text} [SEP] [ASPECT] {aspect} [/ASPECT]"
    
    # Insert markers around aspect in original text
    end_idx = start_idx + len(aspect)
    marked_text = (
        text[:start_idx] + 
        "[ASPECT] " + text[start_idx:end_idx] + " [/ASPECT] " + 
        text[end_idx:]
    )
    return marked_text

In [5]:
# Dataset class - IMPROVED
class DimASRDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Flatten data: one sample per aspect
        self.samples = []
        for item in data:
            text = item['Text']
            aspect_va_list = item['Aspect_VA']
            
            for aspect_item in aspect_va_list:
                aspect = aspect_item['Aspect']
                va = aspect_item['VA']
                v, a = parse_va(va)
                self.samples.append({
                    'id': item['ID'],
                    'text': text,
                    'aspect': aspect,
                    'valence': normalize_va(v),
                    'arousal': normalize_va(a)
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # IMPROVEMENT: Use aspect-highlighted text
        highlighted_text = highlight_aspect_in_text(sample['text'], sample['aspect'])
        
        encoding = self.tokenizer(
            highlighted_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'valence': torch.tensor(sample['valence'], dtype=torch.float),
            'arousal': torch.tensor(sample['arousal'], dtype=torch.float),
            'id': sample['id'],
            'text': sample['text'],
            'aspect': sample['aspect']
        }

class DimASRTestDataset(Dataset):
    """Test dataset without labels"""
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        self.samples = []
        for item in data:
            text = item['Text']
            for aspect in item['Aspect']:
                self.samples.append({
                    'id': item['ID'],
                    'text': text,
                    'aspect': aspect
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        highlighted_text = highlight_aspect_in_text(sample['text'], sample['aspect'])
        
        encoding = self.tokenizer(
            highlighted_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'id': sample['id'],
            'aspect': sample['aspect']
        }

In [6]:
# IMPROVEMENT: Better loss function
class ImprovedVALoss(nn.Module):
    """
    Combines MSE and Smooth L1 loss
    Weights valence more (typically harder to predict)
    """
    def __init__(self, valence_weight=1.2, arousal_weight=0.8, smoothness=0.1):
        super().__init__()
        self.valence_weight = valence_weight
        self.arousal_weight = arousal_weight
        self.smoothness = smoothness
    
    def forward(self, pred, target):
        v_pred, a_pred = pred[:, 0], pred[:, 1]
        v_true, a_true = target[:, 0], target[:, 1]
        
        # MSE loss
        v_mse = F.mse_loss(v_pred, v_true)
        a_mse = F.mse_loss(a_pred, a_true)
        
        # Smooth L1 (less sensitive to outliers)
        v_smooth = F.smooth_l1_loss(v_pred, v_true, beta=self.smoothness)
        a_smooth = F.smooth_l1_loss(a_pred, a_true, beta=self.smoothness)
        
        # Combined
        loss = (
            self.valence_weight * (v_mse + v_smooth) / 2 +
            self.arousal_weight * (a_mse + a_smooth) / 2
        )
        
        return loss

In [7]:
# Model architecture - IMPROVED
class ImprovedVARegressionModel(nn.Module):
    def __init__(self, model_name, hidden_dim, dropout):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        encoder_dim = self.encoder.config.hidden_size
        
        # Larger, deeper regressor
        self.regressor = nn.Sequential(
            nn.Linear(encoder_dim, hidden_dim * 2),  # 768 → 768
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),   # 768 → 384
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), # 384 → 192
            nn.ReLU(),
            nn.Dropout(dropout / 2),  # Less dropout in final layer
            nn.Linear(hidden_dim // 2, 2)           # 192 → 2
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_hidden = outputs.last_hidden_state[:, 0, :]
        return self.regressor(cls_hidden)

In [8]:
# IMPROVEMENT: Detailed evaluation function
def detailed_evaluation(model, dataloader, device):
    """Comprehensive evaluation with multiple metrics"""
    model.eval()
    
    all_v_pred, all_v_true = [], []
    all_a_pred, all_a_true = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            v_true = batch['valence'].cpu().numpy()
            a_true = batch['arousal'].cpu().numpy()
            
            va_pred = model(input_ids, attention_mask).cpu().numpy()
            v_pred, a_pred = va_pred[:, 0], va_pred[:, 1]
            
            all_v_pred.extend(v_pred)
            all_v_true.extend(v_true)
            all_a_pred.extend(a_pred)
            all_a_true.extend(a_true)
    
    # Denormalize for metrics
    v_pred_denorm = denormalize_va(np.array(all_v_pred))
    v_true_denorm = denormalize_va(np.array(all_v_true))
    a_pred_denorm = denormalize_va(np.array(all_a_pred))
    a_true_denorm = denormalize_va(np.array(all_a_true))
    
    # Compute metrics
    v_rmse = np.sqrt(np.mean((v_pred_denorm - v_true_denorm) ** 2))
    a_rmse = np.sqrt(np.mean((a_pred_denorm - a_true_denorm) ** 2))
    overall_rmse = np.sqrt((v_rmse ** 2 + a_rmse ** 2) / 2)
    
    v_mae = np.mean(np.abs(v_pred_denorm - v_true_denorm))
    a_mae = np.mean(np.abs(a_pred_denorm - a_true_denorm))
    
    # Correlation
    v_corr = np.corrcoef(v_pred_denorm, v_true_denorm)[0, 1]
    a_corr = np.corrcoef(a_pred_denorm, a_true_denorm)[0, 1]
    
    # Prediction distribution (check for collapse)
    v_std = np.std(v_pred_denorm)
    a_std = np.std(a_pred_denorm)
    
    metrics = {
        'v_rmse': v_rmse,
        'a_rmse': a_rmse,
        'overall_rmse': overall_rmse,
        'v_mae': v_mae,
        'a_mae': a_mae,
        'v_corr': v_corr,
        'a_corr': a_corr,
        'v_std': v_std,
        'a_std': a_std
    }
    
    return metrics

In [9]:
# Training function
def train_epoch(model, dataloader, optimizer, criterion, device, accumulation_steps=1):
    model.train()
    total_loss = 0
    
    optimizer.zero_grad()
    
    for i, batch in enumerate(tqdm(dataloader, desc='Training')):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        va_true = torch.stack([batch['valence'], batch['arousal']], dim=1).to(device)
        
        va_pred = model(input_ids, attention_mask)
        loss = criterion(va_pred, va_true)
        
        # Gradient accumulation
        loss = loss / accumulation_steps
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps
    
    return total_loss / len(dataloader)

In [10]:
# Load and prepare data
print("Loading training data...")
train_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl')
train_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')

# Convert to Aspect_VA format
train_rest_converted = extract_aspect_va_from_quadruplets(train_rest)
train_laptop_converted = extract_aspect_va_from_quadruplets(train_laptop)

# Combine datasets
train_data = train_rest_converted + train_laptop_converted
print(f"Training samples: {len(train_data)}")

# Load dev data
dev_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_dev_task1.jsonl')
dev_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task1.jsonl')
dev_data = dev_rest + dev_laptop
print(f"Validation samples: {len(dev_data)}")

# Show sample
print("\nSample training data:")
print(json.dumps(train_data[0], indent=2))

Loading training data...
Training samples: 4880
Validation samples: 400

Sample training data:
{
  "ID": "rest16_quad_dev_2",
  "Text": "their sake list was extensive , but we were looking for purple haze , which was n ' t listed but made for us upon request !",
  "Aspect_VA": [
    {
      "Aspect": "sake list",
      "VA": "7.83#8.00"
    }
  ]
}


In [11]:
# Initialize tokenizer and datasets
print("Initializing tokenizer and datasets...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

train_dataset = DimASRDataset(train_data, tokenizer, config.MAX_LENGTH)
dev_dataset = DimASRDataset(dev_data, tokenizer, config.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=config.BATCH_SIZE)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(dev_loader)}")
print(f"Total training samples (flattened): {len(train_dataset)}")
print(f"Total validation samples (flattened): {len(dev_dataset)}")

Initializing tokenizer and datasets...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Training batches: 115
Validation batches: 10
Total training samples (flattened): 7298
Total validation samples (flattened): 615


In [12]:
# Initialize model
print("Initializing model...")
model = ImprovedVARegressionModel(
    config.MODEL_NAME, 
    config.HIDDEN_DIM, 
    config.DROPOUT
).to(config.DEVICE)

# Loss and optimizer
criterion = ImprovedVALoss()
optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

total_steps = len(train_loader) * config.EPOCHS // config.ACCUMULATION_STEPS
warmup_steps = int(total_steps * config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Initializing model...


2026-02-08 16:27:58.197560: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770568078.423875      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770568078.492306      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770568079.024818      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568079.024873      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568079.024877      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Total training steps: 3450
Warmup steps: 517
Model parameters: 110,442,434


In [13]:
# Training loop
print("\nStarting training...\n")
best_rmse = float('inf')
patience_counter = 0

for epoch in range(config.EPOCHS):
    avg_train_loss = train_epoch(
        model, train_loader, optimizer, criterion, 
        config.DEVICE, config.ACCUMULATION_STEPS
    )
    scheduler.step()
    
    # Detailed evaluation
    metrics = detailed_evaluation(model, dev_loader, config.DEVICE)
    
    print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val RMSE - Overall: {metrics['overall_rmse']:.4f} | V: {metrics['v_rmse']:.4f} | A: {metrics['a_rmse']:.4f}")
    print(f"Val MAE  - V: {metrics['v_mae']:.4f} | A: {metrics['a_mae']:.4f}")
    print(f"Val Corr - V: {metrics['v_corr']:.4f} | A: {metrics['a_corr']:.4f}")
    print(f"Pred Std - V: {metrics['v_std']:.4f} | A: {metrics['a_std']:.4f}")
    
    # Check for collapse
    if metrics['v_std'] < 0.5 or metrics['a_std'] < 0.5:
        print("  Warning: Model predictions collapsing!")
    
    # Save best model
    if metrics['overall_rmse'] < best_rmse:
        best_rmse = metrics['overall_rmse']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'rmse': best_rmse,
            'metrics': metrics
        }, f'{config.CHECKPOINT_DIR}/best_model.pt')
        print(f"✅ New best model saved! RMSE: {best_rmse:.4f}")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{config.PATIENCE}")
    
    if patience_counter >= config.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nTraining completed. Best RMSE: {best_rmse:.4f}")


Starting training...



Training:   0%|          | 0/115 [00:00<?, ?it/s]


Epoch 1/30
Train Loss: 1.0022
Val RMSE - Overall: 5.5534 | V: 5.5665 | A: 5.5402
Val MAE  - V: 5.3131 | A: 5.4747
Val Corr - V: -0.2522 | A: -0.0753
Pred Std - V: 0.0499 | A: 0.0556
✅ New best model saved! RMSE: 5.5534


Training:   0%|          | 0/115 [00:00<?, ?it/s]


Epoch 2/30
Train Loss: 0.9897
Val RMSE - Overall: 5.4753 | V: 5.4838 | A: 5.4667
Val MAE  - V: 5.2271 | A: 5.4005
Val Corr - V: -0.2231 | A: -0.0712
Pred Std - V: 0.0494 | A: 0.0545
✅ New best model saved! RMSE: 5.4753


...


Epoch 28/30
Train Loss: 0.0573
Val RMSE - Overall: 0.8864 | V: 0.7658 | A: 0.9925
Val MAE  - V: 0.5477 | A: 0.8185
Val Corr - V: 0.8939 | A: 0.5454
Pred Std - V: 1.6377 | A: 0.6954
No improvement. Patience: 2/7


Training:   0%|          | 0/115 [00:00<?, ?it/s]


Epoch 29/30
Train Loss: 0.0553
Val RMSE - Overall: 0.8247 | V: 0.7426 | A: 0.8993
Val MAE  - V: 0.5360 | A: 0.7216
Val Corr - V: 0.8946 | A: 0.5495
Pred Std - V: 1.5713 | A: 0.6996
No improvement. Patience: 3/7


Training:   0%|          | 0/115 [00:00<?, ?it/s]


Epoch 30/30
Train Loss: 0.0544
Val RMSE - Overall: 0.8438 | V: 0.7598 | A: 0.9201
Val MAE  - V: 0.5423 | A: 0.7446
Val Corr - V: 0.8936 | A: 0.5584
Pred Std - V: 1.6442 | A: 0.7151
No improvement. Patience: 4/7

Training completed. Best RMSE: 0.8196


In [14]:
# Prediction functions
def predict(model, dataloader, device):
    model.eval()
    predictions = {}
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ids = batch['id']
            aspects = batch['aspect']
            
            va_pred = model(input_ids, attention_mask)
            v_pred = denormalize_va(va_pred[:, 0].cpu().numpy())
            a_pred = denormalize_va(va_pred[:, 1].cpu().numpy())
            
            for id_, aspect, v, a in zip(ids, aspects, v_pred, a_pred):
                if id_ not in predictions:
                    predictions[id_] = []
                predictions[id_].append({
                    'Aspect': aspect,
                    'VA': format_va(v, a)
                })
    
    return predictions

def save_predictions(predictions, original_data, output_path):
    """Save predictions in JSONL format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in original_data:
            id_ = item['ID']
            output = {
                'ID': id_,
                'Aspect_VA': predictions.get(id_, [])
            }
            f.write(json.dumps(output) + '\n')
    print(f"Predictions saved to {output_path}")

In [15]:
# Load best model for inference
print("Loading best model...")
checkpoint = torch.load(f'{config.CHECKPOINT_DIR}/best_model.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']} with RMSE {checkpoint['rmse']:.4f}")

Loading best model...
Loaded model from epoch 25 with RMSE 0.8196


In [16]:
# Generate predictions for test sets
print("\nGenerating predictions for test sets...")

# Restaurant test
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task1.jsonl')
test_rest_dataset = DimASRTestDataset(test_rest_data, tokenizer, config.MAX_LENGTH)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)
predictions_rest = predict(model, test_rest_loader, config.DEVICE)
save_predictions(predictions_rest, test_rest_data, f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

# Laptop test
test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task1.jsonl')
test_laptop_dataset = DimASRTestDataset(test_laptop_data, tokenizer, config.MAX_LENGTH)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)
predictions_laptop = predict(model, test_laptop_loader, config.DEVICE)
save_predictions(predictions_laptop, test_laptop_data, f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

print("\n✅ All predictions completed!")


Generating predictions for test sets...


Predicting:   0%|          | 0/24 [00:00<?, ?it/s]

Predictions saved to ./subtask1_outputs/restaurant_test_predictions.jsonl


Predicting:   0%|          | 0/23 [00:00<?, ?it/s]

Predictions saved to ./subtask1_outputs/laptop_test_predictions.jsonl

✅ All predictions completed!


In [17]:
# Sample predictions
print("\nSample predictions:")
print("=" * 80)

test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task1.jsonl')

with open(f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            pred = json.loads(line)
            print(f"\n--- Sample {i+1} ---")
            print(f"Text: {test_rest_data[i]['Text']}")
            print(f"ID: {pred['ID']}")
            for aspect_va in pred['Aspect_VA']:
                print(f"  Aspect: {aspect_va['Aspect']}")
                print(f"  VA Scores: {aspect_va['VA']}")
            print("-" * 50)

print("=" * 80)


Sample predictions:

--- Sample 1 ---
Text: A friend suggested this cafe for a lunch date a while back and I cannot stay away
ID: rest26_aspect_va_test_1
  Aspect: cafe
  VA Scores: 5.11#5.67
--------------------------------------------------

--- Sample 2 ---
Text: The beer selection is second to none , but this time we had one drink and decided to spend our money elsewhere for dinner
ID: rest26_aspect_va_test_2
  Aspect: beer selection
  VA Scores: 4.88#5.47
--------------------------------------------------

--- Sample 3 ---
Text: It was pretty bland for my liking - in complete contrast to the sausage and pepper pizza - and the pepperoni was pretty sparse
ID: rest26_aspect_va_test_3
  Aspect: pepper pizza
  VA Scores: 3.63#6.15
  Aspect: pepperoni
  VA Scores: 3.61#6.04
  Aspect: sausage
  VA Scores: 3.46#6.30
--------------------------------------------------

--- Sample 4 ---
Text: Half price beer during a generous happy hour , a huge deck that welcomes dogs , and delicious key l

In [ ]:
# Performance summary
print("\n" + "=" * 80)
print("PERFORMANCE SUMMARY")
print("=" * 80)
print(f"\nBest Model Metrics:")
print(f"  Overall RMSE: {checkpoint['metrics']['overall_rmse']:.4f}")
print(f"  Valence RMSE: {checkpoint['metrics']['v_rmse']:.4f}")
print(f"  Arousal RMSE: {checkpoint['metrics']['a_rmse']:.4f}")
print(f"  Valence MAE:  {checkpoint['metrics']['v_mae']:.4f}")
print(f"  Arousal MAE:  {checkpoint['metrics']['a_mae']:.4f}")
print(f"  Valence Corr: {checkpoint['metrics']['v_corr']:.4f}")
print(f"  Arousal Corr: {checkpoint['metrics']['a_corr']:.4f}")
print(f"\nPrediction Statistics:")
print(f"  Valence Std:  {checkpoint['metrics']['v_std']:.4f}")
print(f"  Arousal Std:  {checkpoint['metrics']['a_std']:.4f}")
print("\n" + "=" * 80)